# Model Architecture and Output Neuron Walkthrough: custom 3CNN vs. ResNet18 (1 channel)

This notebook is about understanding the model architecture and the very last layer, not about plots.

It uses the same real-data-only pipeline as in `semi_supervised_learing.ipynb`:
- merged real labels from the local CSV sources
- no synthetic data
- the same modulo-4 split
- the same fold-based training setup for both models

Main questions:
- What does each architecture look like from input to output?
- Where is the last feature vector (`pre_logits`)?
- How is the dense output head organized?
- What are the two output neurons for `no_class` and `erp_class`?
- For one concrete validation sample, which values enter each output neuron before everything is summed into the final logit?

For every output neuron `j`, the dense head computes:

`logit_j = bias_j + sum_i weight[j, i] * pre_logits[i]`

ResNet18 note:
- the first convolution is adapted to one input channel
- the pretrained RGB filters are converted with `sum(...; dims = 3)`
- this keeps the pretrained stem useful for grayscale ERP images


In [1]:
import Pkg

model_test_dir = if isfile(joinpath(pwd(), "Project.toml")) && isfile(joinpath(pwd(), "semi_supervised_learing.ipynb"))
    pwd()
else
    candidate = joinpath(pwd(), "notebooks", "model_test")
    @assert isfile(joinpath(candidate, "Project.toml")) "Could not locate notebooks/model_test from current working directory."
    candidate
end

cd(model_test_dir)
Pkg.activate(model_test_dir)

include(joinpath(pwd(), "test_outputlayer_helpers.jl"))
using .TestOutputLayerHelpers
using PrettyTables

table_kwargs = TestOutputLayerHelpers.TABLE_KWARGS

println("Model-test directory: ", pwd())
println("CUDA available: ", TestOutputLayerHelpers.USE_CUDA)


  Activating project at `~/Dokumente/BA2/notebooks/model_test`


Model-test directory: /home/benjamin/Dokumente/BA2/notebooks/model_test
CUDA available: true


In [2]:
ctx = run_outputlayer_analysis(
    analysis_fold = 1,
    data_split_seed = 20260308,
    fold_seed = 20260220,
    cnn3_epochs = 6,
    resnet_epochs = 4,
    cnn3_lr = 1f-3,
    resnet_lr = 1f-4,
    cnn3_batchsize = 32,
    resnet_batchsize = 16,
)

nothing


[ Info: Running output-layer analysis for cnn_3conv
[ Info: cnn_3conv | epoch 1/6 | train_loss=0.62008
[ Info: cnn_3conv | epoch 2/6 | train_loss=0.61129
[ Info: cnn_3conv | epoch 3/6 | train_loss=0.60393
[ Info: cnn_3conv | epoch 4/6 | train_loss=0.59918
[ Info: cnn_3conv | epoch 5/6 | train_loss=0.59795
[ Info: cnn_3conv | epoch 6/6 | train_loss=0.59585
[ Info: Running output-layer analysis for resnet18_pretrained_1ch
[ Info: resnet18_pretrained_1ch | epoch 1/4 | train_loss=0.64361
[ Info: resnet18_pretrained_1ch | epoch 2/4 | train_loss=0.16179
[ Info: resnet18_pretrained_1ch | epoch 3/4 | train_loss=0.09932
[ Info: resnet18_pretrained_1ch | epoch 4/4 | train_loss=0.07213


## Data Pipeline and Training Context

The next tables only establish the shared training context for both models.
The real focus of the notebook starts after this section: architecture, output neurons, and the exact last-layer values.


In [3]:
println("\nLabel summary (before modulo split):")
pretty_table(ctx.label_summary_df; table_kwargs...)

println("\nInput data summary (after modulo split):")
pretty_table(ctx.input_stats_df; table_kwargs...)

println("\nVariant summary (after modulo split):")
pretty_table(ctx.variant_stats_df; table_kwargs...)

println("\nSelected analysis fold summary:")
pretty_table(ctx.fold_summary_df; table_kwargs...)

println("\nSingle-fold model summary:")
pretty_table(ctx.analysis_summary_df; table_kwargs...)

println("\nResNet18 1-channel configuration:")
pretty_table(ctx.resnet_config_df; table_kwargs...)

nothing



Label summary (before modulo split):
┌──────────────┬───────┐
│ binary_label │ count │
│        Int64 │ Int64 │
├──────────────┼───────┤
│            0 │   452 │
│            1 │    48 │
└──────────────┴───────┘

Input data summary (after modulo split):
┌─────────────────────────┬───────┐
│                  metric │ value │
│                  String │ Int64 │
├─────────────────────────┼───────┤
│         original_labels │   500 │
│ augmented_samples_total │   644 │
│    unique_source_groups │   500 │
│        no_class_samples │   452 │
│           class_samples │   192 │
└─────────────────────────┴───────┘

Variant summary (after modulo split):
┌──────────────┬────────────┬───────┬─────────────┬────────────┬────────────┐
│ binary_label │    variant │ count │ mean_trials │ min_trials │ max_trials │
│        Int64 │     String │ Int64 │     Float64 │      Int64 │      Int64 │
├──────────────┼────────────┼───────┼─────────────┼────────────┼────────────┤
│            0 │ mod4_keep1 │   10

## Architecture Trace and Output-Layer Anatomy

The architecture trace follows the real forward path of one validation image:
- `feature_maps`: spatial feature extraction
- `pre_logits`: the last transformation into the final feature vector
- `head`: the dense output layer with two output neurons

Important interpretation:
- `cnn_3conv` ends with a feature vector of length `64`
- `resnet18_pretrained_1ch` ends with a feature vector of length `512`
- the dense head therefore has shape `(2, 64)` for the 3CNN and `(2, 512)` for ResNet18
- neuron 1 corresponds to `no_class`, neuron 2 corresponds to `erp_class`


In [4]:
cnn_run = ctx.analysis_runs["cnn_3conv"]
resnet_run = ctx.analysis_runs["resnet18_pretrained_1ch"]

println("\nArchitecture trace: cnn_3conv")
pretty_table(model_architecture_trace_df(cnn_run); table_kwargs...)

println("\nArchitecture trace: resnet18_pretrained_1ch")
pretty_table(model_architecture_trace_df(resnet_run); table_kwargs...)

println("\nLast-layer overview: cnn_3conv")
pretty_table(last_layer_overview_df(cnn_run); table_kwargs...)

println("\nLast-layer overview: resnet18_pretrained_1ch")
pretty_table(last_layer_overview_df(resnet_run); table_kwargs...)

println("\nOutput-neuron summary: cnn_3conv")
pretty_table(output_neuron_summary_df(cnn_run); table_kwargs...)

println("\nOutput-neuron summary: resnet18_pretrained_1ch")
pretty_table(output_neuron_summary_df(resnet_run); table_kwargs...)

nothing



Architecture trace: cnn_3conv
┌──────────────┬───────────┬─────────────────────┬─────────────────┬───────────────────────────────────────┐
│        stage │ layer_idx │          layer_name │    output_shape │                                  note │
│       String │     Int64 │              String │          String │                                String │
├──────────────┼───────────┼─────────────────────┼─────────────────┼───────────────────────────────────────┤
│        input │         0 │           ERP image │  (64, 64, 1, 1) │            one image as H x W x C x N │
│ feature_maps │         1 │  Conv(3x3, 1 => 16) │ (64, 64, 16, 1) │                                       │
│ feature_maps │         2 │          NNlib.relu │ (64, 64, 16, 1) │                                       │
│ feature_maps │         3 │             MaxPool │ (32, 32, 16, 1) │                                       │
│ feature_maps │         4 │ Conv(3x3, 16 => 32) │ (32, 32, 32, 1) │                             

## Strongest Weights in the Output Layer

These tables describe the trained output neurons themselves.

How to read them:
- one output neuron has one full weight vector
- a large positive weight pushes that neuron upward when the corresponding feature is positive
- a large negative weight pushes that neuron downward when the corresponding feature is positive
- this is still model structure, not yet a concrete sample


In [5]:
println("\nTop output-layer weights: cnn_3conv | neuron 1 = no_class")
pretty_table(output_neuron_weight_ranking_df(cnn_run, 1; top_k = 16); table_kwargs...)

println("\nTop output-layer weights: cnn_3conv | neuron 2 = erp_class")
pretty_table(output_neuron_weight_ranking_df(cnn_run, 2; top_k = 16); table_kwargs...)

println("\nTop output-layer weights: resnet18_pretrained_1ch | neuron 1 = no_class")
pretty_table(output_neuron_weight_ranking_df(resnet_run, 1; top_k = 24); table_kwargs...)

println("\nTop output-layer weights: resnet18_pretrained_1ch | neuron 2 = erp_class")
pretty_table(output_neuron_weight_ranking_df(resnet_run, 2; top_k = 24); table_kwargs...)

nothing



Top output-layer weights: cnn_3conv | neuron 1 = no_class
┌────────────┬────────────┬─────────────┬───────────┬────────────┬─────────────┐
│ neuron_idx │ class_name │ feature_idx │    weight │ abs_weight │ weight_sign │
│      Int64 │     String │       Int64 │   Float64 │    Float64 │      String │
├────────────┼────────────┼─────────────┼───────────┼────────────┼─────────────┤
│          1 │   no_class │          23 │  0.308652 │   0.308652 │    positive │
│          1 │   no_class │          36 │  0.303043 │   0.303043 │    positive │
│          1 │   no_class │          12 │  0.301236 │   0.301236 │    positive │
│          1 │   no_class │          57 │  0.270566 │   0.270566 │    positive │
│          1 │   no_class │          16 │ -0.268484 │   0.268484 │    negative │
│          1 │   no_class │          38 │ -0.267781 │   0.267781 │    negative │
│          1 │   no_class │          44 │ -0.263366 │   0.263366 │    negative │
│          1 │   no_class │          62 │ -0.26176

## Concrete Example: One Validation Sample

A single validation sample is selected below.

First we print a compact summary per output neuron:
- `bias`
- sum of all feature contributions
- final manual logit
- stored model logit
- softmax probability

After that, the notebook prints the values before the final sum inside each output neuron.


In [6]:
println("\nCandidate validation rows from both models:")
pretty_table(ctx.interesting_compare_df; table_kwargs...)

selected_val_row = first(ctx.interesting_val_rows)

selected_instance_df = cnn_run.predictions[[selected_val_row], [
    :val_row,
    :sample_id,
    :group_id,
    :image_id,
    :sort_var,
    :variant,
    :true_name,
]]

println("\nSelected validation sample:")
pretty_table(selected_instance_df; table_kwargs...)

println("\nPer-neuron summary for the selected sample: cnn_3conv")
pretty_table(instance_neuron_summary_df(cnn_run, selected_val_row); table_kwargs...)

println("\nPer-neuron summary for the selected sample: resnet18_pretrained_1ch")
pretty_table(instance_neuron_summary_df(resnet_run, selected_val_row); table_kwargs...)

nothing



Candidate validation rows from both models:
┌─────────┬───────────┬───────────────┬────────────┬───────────┬─────────────────────┬──────────────────────┬────────────────────┬─────────────────────┬────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬──────────────────┐
│ val_row │ sample_id │      sort_var │    variant │ true_name │ cnn3_logit_no_class │ cnn3_logit_erp_class │ cnn3_prob_no_class │ cnn3_prob_erp_class │ cnn3_pred_name │ resnet_logit_no_class │ resnet_logit_erp_class │ resnet_prob_no_class │ resnet_prob_erp_class │ resnet_pred_name │
│   Int64 │     Int64 │        String │     String │    String │             Float64 │              Float64 │            Float64 │             Float64 │         String │               Float64 │                Float64 │              Float64 │               Float64 │           String │
├─────────┼───────────┼───────────────┼────────────┼───────────┼─────────────────────┼──────────────

## Before the Final Sum: Values Inside Each Output Neuron

Each row in the next tables is one feature entering one specific output neuron.

Columns:
- `feature_value`: one entry of the `pre_logits` vector
- `weight`: the matching parameter of the selected output neuron
- `contribution`: `feature_value * weight`
- `logit_if_only_this_feature`: `bias + contribution`
- `running_logit_display_order`: partial sum over the rows as they are printed

So this is the exact printout of the values that exist before the final aggregation into one logit per class.


In [7]:
cnn_no_class_neuron_df = instance_neuron_breakdown_df(
    cnn_run,
    selected_val_row,
    1;
    sort_by = :feature_idx,
)

cnn_erp_class_neuron_df = instance_neuron_breakdown_df(
    cnn_run,
    selected_val_row,
    2;
    sort_by = :feature_idx,
)

println("\nSelected sample | cnn_3conv | output neuron = no_class | all 64 values before summation")
pretty_table(cnn_no_class_neuron_df; table_kwargs...)

println("\nSelected sample | cnn_3conv | output neuron = erp_class | all 64 values before summation")
pretty_table(cnn_erp_class_neuron_df; table_kwargs...)

nothing



Selected sample | cnn_3conv | output neuron = no_class | all 64 values before summation
┌────────────┬────────────┬─────────────┬───────────────┬─────────────┬──────────────┬───────────────────┬────────────┬──────────────────┬───────────┬────────────────────────────┬─────────────────────────────┬───────────────────────┬─────────────┐
│ neuron_idx │ class_name │ feature_idx │ feature_value │      weight │ contribution │ abs_feature_value │ abs_weight │ abs_contribution │      bias │ logit_if_only_this_feature │ running_logit_display_order │ total_feature_contrib │ final_logit │
│      Int64 │     String │       Int64 │       Float64 │     Float64 │      Float64 │           Float64 │    Float64 │          Float64 │   Float64 │                    Float64 │                     Float64 │               Float64 │     Float64 │
├────────────┼────────────┼─────────────┼───────────────┼─────────────┼──────────────┼───────────────────┼────────────┼──────────────────┼───────────┼─────────────────

In [8]:
resnet_no_class_top_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    1;
    sort_by = :abs_contribution,
    top_k = 32,
)

resnet_erp_class_top_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    2;
    sort_by = :abs_contribution,
    top_k = 32,
)

# Full 512-entry tables are kept in these variables for direct inspection.
resnet_no_class_full_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    1;
    sort_by = :feature_idx,
)

resnet_erp_class_full_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    2;
    sort_by = :feature_idx,
)

println("\nSelected sample | resnet18_pretrained_1ch | output neuron = no_class | top 32 values by |contribution|")
pretty_table(resnet_no_class_top_df; table_kwargs...)

println("\nSelected sample | resnet18_pretrained_1ch | output neuron = erp_class | top 32 values by |contribution|")
pretty_table(resnet_erp_class_top_df; table_kwargs...)

println("\n`resnet_no_class_full_df` and `resnet_erp_class_full_df` contain all 512 values in original feature order.")
nothing



Selected sample | resnet18_pretrained_1ch | output neuron = no_class | top 32 values by |contribution|
┌────────────┬────────────┬─────────────┬───────────────┬────────────┬──────────────┬───────────────────┬────────────┬──────────────────┬────────────┬────────────────────────────┬─────────────────────────────┬───────────────────────┬─────────────┐
│ neuron_idx │ class_name │ feature_idx │ feature_value │     weight │ contribution │ abs_feature_value │ abs_weight │ abs_contribution │       bias │ logit_if_only_this_feature │ running_logit_display_order │ total_feature_contrib │ final_logit │
│      Int64 │     String │       Int64 │       Float64 │    Float64 │      Float64 │           Float64 │    Float64 │          Float64 │    Float64 │                    Float64 │                     Float64 │               Float64 │     Float64 │
├────────────┼────────────┼─────────────┼───────────────┼────────────┼──────────────┼───────────────────┼────────────┼──────────────────┼────────────┼──